# Phase 1: local Qwen3 hint test
This notebook runs the same pinned small model used in Phases 2–3. It saves exact prompt and generated token IDs. It is not a completed experiment; no results are prefilled. A 20-question pilot means 100 responses (two baselines plus three hints each).

## Setup
Select **Runtime → Change runtime type → T4 GPU**. Upload the project ZIP containing a top-level `hint-faithfulness/` folder. You may select a saved results ZIP at the same time; the notebook restores it after the project, so you do not need to repack archives. Include local Phase 1 records when continuing an experiment. Do not include credentials or virtual environments. Free GPU availability is not guaranteed.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, zipfile

# Select the project ZIP and any saved results ZIP together.
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
if not archives:
    raise ValueError("Upload the project ZIP, optionally with a saved results ZIP")
root = Path("/content").resolve()
def contains_project(name):
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        return "hint-faithfulness/src/run.py" in archive.namelist()
# Install project files first; restore saved data and human labels afterward.
archives.sort(key=lambda name: 0 if contains_project(name) else 1)
for name in archives:
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        for member in archive.infolist():
            destination = (root / member.filename).resolve()
            if not destination.is_relative_to(root):
                raise ValueError("Archive path escapes /content")
        archive.extractall(root)
project = root / "hint-faithfulness"
if not (project / "src" / "run.py").exists():
    raise ValueError("Project ZIP must contain hint-faithfulness/src/run.py")
os.chdir(project)


In [ ]:
!python -m pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before loading weights"
print(torch.cuda.get_device_name(0))
!python -m unittest discover -s tests -q


## Predict before running
Write your own prediction below. This is a dated expectation, not a result. Keep it when the outcome disagrees.

In [ ]:
from datetime import datetime, timezone
PREDICTION = ""  # Fill in your prediction before running the experiment.
assert PREDICTION.strip(), "Write a prediction first"
with Path("LOG.md").open("a") as log:
    log.write(f"\n\n{datetime.now(timezone.utc).isoformat()} — prediction\n{PREDICTION}\n")


In [ ]:
!python -m src.run --backend local --questions data/questions.jsonl --output data/runs/small.jsonl --limit 20 --max-tokens 4096 --device cuda


## Read the transcripts
Read the complete pilot responses before interpreting any rate. Check thought/final-answer separation, truncation, whether hints are actually wrong, and the final answer parser. The code below writes a readable file; open it from Colab’s Files sidebar.

In [ ]:
from src.common import read_jsonl
rows = read_jsonl("data/runs/small.jsonl")
Path("results").mkdir(exist_ok=True)
with Path("results/pilot_transcripts.md").open("w") as stream:
    for row in rows:
        stream.write(f"\n## {row['question_id']} — {row['hint_type']} / {row.get('baseline_repeat')}\n")
        stream.write(row["prompt"] + "\n\nThinking:\n" + row.get("reasoning", "") + "\n\nFinal:\n" + row.get("content", "") + "\n")
print("Responses:", len(rows))
print("Statuses:", {status: sum(r["status"] == status for r in rows) for status in {r["status"] for r in rows}})


## Extend to 150 questions
After transcript inspection, the same command resumes the existing file and adds the remaining questions. If token limits or sampling settings need changing, choose a new output file so experiments stay distinct. Judge calibration and human labels follow the README; a notebook does not supply human judgments.

In [ ]:
PILOT_REVIEWED = False  # Set True only after reading the pilot and checking the pipeline.
assert PILOT_REVIEWED
!python -m src.run --backend local --questions data/questions.jsonl --output data/runs/small.jsonl --limit 150 --max-tokens 4096 --device cuda


## Save your work
Download results before the Colab session ends. The archive includes data, logs, and experiment outputs, but excludes downloaded weights and credentials.

In [ ]:
from google.colab import files
with zipfile.ZipFile("research-results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("data"), Path("results")):
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file():
                    archive.write(item, Path("hint-faithfulness") / item)
    if Path("LOG.md").exists():
        archive.write("LOG.md", "hint-faithfulness/LOG.md")
files.download("research-results.zip")
